# 実験: Exp-BFly-A（全バタフライ水準追加）

**目的**: B2〜B7 全ての水準を全行に特徴量として追加した場合に CS IC が改善するか確認する。

## ベースライン IC（目標）
| Horizon | Global IC | CS IC | Train IC | Gap |
|---------|-----------|-------|----------|-----|
| 3d | 0.373 | **0.304** | 0.521 | 0.148 |
| 5d | 0.418 | **0.346** | 0.573 | 0.154 |

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.processing        import load_and_clean_data
from src.features_rv       import generate_rv_features
from src.pooling_butterfly import pool_butterfly_data
from src.modeling          import walk_forward_with_model, summarize_ic

START_DATE = '2024-01-01'
INSTRUMENT_INDICES = set(range(2, 8))

print('Loading data...')
df_raw = load_and_clean_data('../data/BOJ_data.xlsx', '../data/BOJ_meeting_history.csv')
df_rv  = generate_rv_features(df_raw)

# 1. Baseline Data
df_fly_baseline = pool_butterfly_data(df_rv)

print(f'Data: {df_raw["Date"].min().date()} → {df_raw["Date"].max().date()}')
print(f'Baseline shape: {df_fly_baseline.shape}')

## 1. ベースライン IC の再現

In [ ]:
print('Running Baseline Walk-forward...')
res_3d_b, _, _, _, _ = walk_forward_with_model(df_fly_baseline, 'Target_3d_norm', START_DATE)
res_5d_b, _, _, _, _ = walk_forward_with_model(df_fly_baseline, 'Target_5d_norm', START_DATE)

ic3_b = summarize_ic(res_3d_b, instrument_indices=INSTRUMENT_INDICES)
ic5_b = summarize_ic(res_5d_b, instrument_indices=INSTRUMENT_INDICES)

print('\n=== Baseline IC (RV Butterfly) ===')
print(f'               3d        5d')
print(f'Global IC  : {ic3_b["ic_all"]:>8.4f}  {ic5_b["ic_all"]:>8.4f}')
print(f'CS IC      : {ic3_b["cs_ic"]:>8.4f}  {ic5_b["cs_ic"]:>8.4f}')
print(f'Train IC   : {ic3_b["train_ic"]:>8.4f}  {ic5_b["train_ic"]:>8.4f}')
print(f'Gap        : {ic3_b["gap"]:>8.4f}  {ic5_b["gap"]:>8.4f}')

## 2. Exp-BFly-A (全B水準追加) の実装

ノートブック内でインラインプーリングを実装し、B2〜B7 level を全行に持たせる。

In [ ]:
def pool_butterfly_data_exp_a(feat_df: pd.DataFrame) -> pd.DataFrame:
    df = feat_df.copy()
    
    # B2_level〜B7_level を全行に引き継ぐため、melt 前に列として追加
    b_level_cols = []
    for n in range(2, 8):
        col_name = f'B{n}_level'
        df[col_name] = df[f'B{n}']
        b_level_cols.append(col_name)
        
    fly_raw_cols = [f'B{n}' for n in range(2, 8)]
    id_cols = [c for c in df.columns if c not in fly_raw_cols]
    
    pooled = df.melt(
        id_vars=id_cols,
        value_vars=fly_raw_cols,
        var_name='Rate_Label',
        value_name='Rate_Value',
    )
    
    pooled['Meeting_Index'] = pooled['Rate_Label'].str.extract('(\d+)').astype(int)
    pooled = pooled.sort_values(['Rate_Label', 'Date']).reset_index(drop=True)

    # 目的変数の生成
    for h in [1, 3, 5]:
        pooled[f'Target_{h}d'] = (
            pooled.groupby('Rate_Label')['Rate_Value'].shift(-h) - pooled['Rate_Value']
        )
    for h in [3, 5]:
        instr_std = pooled.groupby('Rate_Label')[f'Target_{h}d'].transform('std')
        pooled[f'Target_{h}d_std']  = instr_std
        pooled[f'Target_{h}d_norm'] = pooled[f'Target_{h}d'] / instr_std
    
    # Feature selection (matching src/pooling_butterfly.py names)
    imputed_cols_fly = []
    for n in range(2, 8):
        for col in [f'M{n-1}_is_imputed', f'M{n}_is_imputed', f'M{n+1}_is_imputed']:
            if col in pooled.columns and col not in imputed_cols_fly:
                imputed_cols_fly.append(col)

    basic_cols   = ['Meeting_Index', 'is_post_mpm', 'Days_to_MPM', 'Actual_Policy_Rate']
    anchor_cols  = ['M1_spread', 'M1_frac_diff', 'Slope_M1M8', 'Slope_M1M8_frac_diff']
    fly_fd_cols  = [f'B{n}_frac_diff' for n in range(2, 8)]
    ext_fd_cols  = ['USDJPY_frac_diff', 'JGB_Future_frac_diff', 'Nikkei225_frac_diff', 'DXY_frac_diff']
    
    target_cols = ['Date', 'Target_1d_norm', 'Target_3d_norm', 'Target_5d_norm', 
                   'Target_1d_std', 'Target_3d_std', 'Target_5d_std']
    
    final_cols = (
        basic_cols
        + b_level_cols
        + anchor_cols
        + fly_fd_cols
        + imputed_cols_fly
        + ext_fd_cols
        + target_cols
    )
    
    available = [c for c in final_cols if c in pooled.columns]
    return pooled[available]

df_fly_exp_a = pool_butterfly_data_exp_a(df_rv)
print(f'Exp-A shape: {df_fly_exp_a.shape}')
print(f'Features: {len([c for c in df_fly_exp_a.columns if c not in ["Date", "Target_1d_norm", "Target_3d_norm", "Target_5d_norm", "Target_1d_std", "Target_3d_std", "Target_5d_std", "is_post_mpm"]])}')
print(f'Feature names: {[c for c in df_fly_exp_a.columns if c not in ["Date", "Target_1d_norm", "Target_3d_norm", "Target_5d_norm", "Target_1d_std", "Target_3d_std", "Target_5d_std", "is_post_mpm"]]}')

## 3. Exp-BFly-A の実行

In [ ]:
print('Running Exp-BFly-A Walk-forward...')
res_3d_a, mdl_3d_a, _, _, _ = walk_forward_with_model(df_fly_exp_a, 'Target_3d_norm', START_DATE)
res_5d_a, mdl_5d_a, _, _, _ = walk_forward_with_model(df_fly_exp_a, 'Target_5d_norm', START_DATE)

ic3_a = summarize_ic(res_3d_a, instrument_indices=INSTRUMENT_INDICES)
ic5_a = summarize_ic(res_5d_a, instrument_indices=INSTRUMENT_INDICES)

print('\n=== Exp-BFly-A IC (All B Levels) ===')
print(f'               3d        5d')
print(f'Global IC  : {ic3_a["ic_all"]:>8.4f}  {ic5_a["ic_all"]:>8.4f}')
print(f'CS IC      : {ic3_a["cs_ic"]:>8.4f}  {ic5_a["cs_ic"]:>8.4f}')
print(f'Train IC   : {ic3_a["train_ic"]:>8.4f}  {ic5_a["train_ic"]:>8.4f}')
print(f'Gap        : {ic3_a["gap"]:>8.4f}  {ic5_a["gap"]:>8.4f}')

## 4. 比較と可視化

In [ ]:
results_df = pd.DataFrame({
    'Metric': ['3d Global IC', '3d CS IC', '3d Train IC', '3d Gap', 
               '5d Global IC', '5d CS IC', '5d Train IC', '5d Gap'],
    'Baseline': [ic3_b['ic_all'], ic3_b['cs_ic'], ic3_b['train_ic'], ic3_b['gap'],
                 ic5_b['ic_all'], ic5_b['cs_ic'], ic5_b['train_ic'], ic5_b['gap']],
    'Exp-BFly-A': [ic3_a['ic_all'], ic3_a['cs_ic'], ic3_a['train_ic'], ic3_a['gap'],
                   ic5_a['ic_all'], ic5_a['cs_ic'], ic5_a['train_ic'], ic5_a['gap']]
})
results_df['Delta'] = results_df['Exp-BFly-A'] - results_df['Baseline']
print(results_df.to_string(index=False))

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Butterfly Global IC by Fold: Baseline vs Exp-BFly-A', fontsize=12, fontweight='bold')

for i, (ic_b, ic_a, title) in enumerate([(ic3_b, ic3_a, '3d'), (ic5_b, ic5_a, '5d')]):
    ax = axes[i]
    folds = sorted(ic_b['ic_by_fold'].keys())
    b_vals = [ic_b['ic_by_fold'][f] for f in folds]
    a_vals = [ic_a['ic_by_fold'][f] for f in folds]
    
    x = np.arange(len(folds))
    width = 0.35
    
    ax.bar(x - width/2, b_vals, width, label='Baseline', color='#cccccc')
    ax.bar(x + width/2, a_vals, width, label='Exp-BFly-A', color='#ff7f0e' if i==0 else '#d62728')
    
    ax.set_title(f'{title} Horizon')
    ax.set_xticks(x)
    ax.set_xticklabels([str(f) for f in folds])
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig('exp_bfly_a_results.png')
plt.show()

## 5. 特徴量重要度の確認

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Importance (gain, last fold, top 20) - Exp-BFly-A', fontsize=12, fontweight='bold')

for ax, model, title, color in [
        (axes[0], mdl_3d_a, 'Butterfly 3d', '#ff7f0e'),
        (axes[1], mdl_5d_a, 'Butterfly 5d', '#d62728')]:
    imp = pd.DataFrame({'feature': model.feature_name(),
                        'gain':    model.feature_importance(importance_type='gain')})
    imp = imp.sort_values('gain', ascending=False).head(20)
    ax.barh(imp['feature'][::-1], imp['gain'][::-1], color=color, edgecolor='white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Gain'); ax.tick_params(labelsize=8); ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()